# CodeAct over CLDK

In `cldk-mcp.ipynb`, we asked CLDK four questions by hand and then served them as four MCP tools.

Here, we give CLDK to an agent in the **CodeAct** style from lecture 4:

- The model writes Python that calls CLDK.
- The harness runs that Python.
- Whatever the code prints goes back to the model as the next observation.

The agent has **no tools**. It has two things:

- the CLDK analysis object `an`, already loaded, and
- a system prompt that teaches it how to use `an`.

Because each reply is a program, one reply can do several CLDK calls at once. With MCP, each call was a separate tool call.

**Plan**

1. Load CLDK.
2. Build the kernel that runs the model's code.
3. Write the prompt that teaches CLDK, and check it.
4. Connect the model.
5. Run the agent on one true positive and one false positive.
6. See what the agent's code can reach.

In [1]:
import importlib.metadata
import os
import subprocess
import sys

# CLDK needs to know which Python to analyse with. Point it at uv's Python 3.12.
os.environ["SYSTEM_PYTHON"] = subprocess.run(
    ["uv", "python", "find", "3.12"], capture_output=True, text=True
).stdout.strip()

# Confirm we are in this folder's .venv and see which CLDK we have.
print(sys.executable)
print("cldk", importlib.metadata.version("cldk"))

/Users/rkrsn/COMSE6998-019/lectures/lecture-6--2026-10-16/.venv/bin/python
cldk 2.0.0


## 1. Load CLDK

This is the same call as in `cldk-mcp.ipynb`.

It reuses the cache in `cldk-demo/owasp/.codeanalyzer`, so it takes seconds. Without the cache, it analyses all 3,710 callables, which takes minutes.

In [2]:
import time

from cldk import CLDK

t = time.time()

# Build (or load from cache) the analysis of the OWASP checkout.
# "program_dependency_graph" gives us symbols, the call graph and per-function dataflow.
an = CLDK.python(project_path="cldk-demo/owasp", analysis_level="program_dependency_graph")

print(len(an.get_callables_overview()), "callables,", f"{time.time() - t:.0f} s")

3710 callables, 11 s


## 2. The kernel

The kernel is the part that runs the model's code. It is lecture 4's minimal kernel (notes, section 4.4) with two small changes:

- **`an` is already there.** Lecture 4's namespace started empty. Here we put `an` (CLDK) and `nx` (networkx) in it, so the model can use them straight away.
- **Output is capped.** A script could print a whole call graph. We keep the first 6,000 characters and add a note saying the rest was cut.

Everything else is the same as lecture 4:

- One namespace is used for the whole run, so variables survive from one script to the next.
- An error does not crash the harness. The traceback is sent back to the model, which can fix its code.

In [3]:
import io, re, traceback
from contextlib import redirect_stdout
import networkx as nx

# Upper bound on the characters of output returned to the model per script.
# Keeps one observation from filling the model's context window.
MAX_OBSERVATION = 6000


def new_namespace() -> dict:
    """A fresh namespace holding only the analysis and networkx."""
    # The harness, not the model, seeds the namespace. `an` is the CLDK analysis
    # attached in section 1, and `nx` is for graph searches. Every episode gets
    # its own dict, so variables from one run never leak into the next.
    return {"an": an, "nx": nx}


def extract(reply: str) -> str | None:
    """The first fenced Python block of a reply, or None when the reply has no code."""
    # Enforces the action format: one fenced ```python block per reply.
    # re.S lets "." match newlines, so the block may span many lines.
    # The non-greedy (.*?) stops at the first closing fence.
    m = re.search(r"```python\n(.*?)```", reply, re.S)
    # No block means the reply is the final answer, which ends the loop.
    return m.group(1) if m else None


def run(script: str, namespace: dict) -> str:
    """Run one script in the namespace; return what it printed, or its traceback."""
    # Capture stdout: whatever the script prints becomes the observation.
    out = io.StringIO()
    try:
        with redirect_stdout(out):
            # exec with the persistent namespace: variables and imports defined
            # by one script are still there for the next turn.
            exec(script, namespace)
    except Exception:
        # An error does not stop the harness. The traceback is appended to the
        # output, so the model sees it as the observation and can fix its script.
        out.write(traceback.format_exc())
    # Tell the model explicitly when nothing was printed, rather than send an empty string.
    text = out.getvalue() or "(the script printed nothing)"
    # Cap the observation and leave a marker, so the model knows output was cut.
    if len(text) > MAX_OBSERVATION:
        text = text[:MAX_OBSERVATION] + f"\n[output cut at {MAX_OBSERVATION} of {len(text)} characters]"
    return text

### Test the kernel by hand

Before involving a model, we run three scripts ourselves:

1. Find the flagged line and save its signature in `sig`.
2. Use `sig` again. It still exists, because the namespace persists.
3. Make a mistake on purpose: the bare name `init` matches 1,230 functions. The kernel returns CLDK's error as text.

In [18]:
ns = new_namespace()

# Script 1: find the function that holds line 46, and keep its signature in `sig`.
print(
    run(
        'hit = an.locate("testcode/BenchmarkTest00099.py", 46)\n'
        "sig = hit.callable.signature\n"
        "print(sig)",
        ns,
    )
)

# Script 2: `sig` is still defined, because the namespace carries over.
print(
    run(
        'print(len(an.get_ddg(sig).edges), "data-dependence edges")',
        ns,
    )
)

# Script 3: an ambiguous name. The error comes back as text; we print its last line.
error = run('an.get_ddg("init")', ns)
print(error.strip().splitlines()[-1])

testcode.BenchmarkTest00099.init.BenchmarkTest00099_post

49 data-dependence edges

cldk.utils.exceptions.exceptions.AmbiguousName: 'init' is ambiguous: 1230 callables match. Narrow it with in_module= or more of the dotted path. Matches: 'testcode.BenchmarkTest00001.init'; 'testcode.BenchmarkTest00002.init'; 'testcode.BenchmarkTest00003.init'; 'testcode.BenchmarkTest00004.init'; 'testcode.BenchmarkTest00005.init'; ... and 1225 more


## 3. Teach the agent CLDK

The model has never seen CLDK 2.0, so the system prompt has to teach it. The prompt has four parts:

- **How to act.** Reply with one ```` ```python ```` block. Print what you want to see. Reply without code when you are done.
- **An API card.** The five CLDK calls the task needs, and what each one returns.
- **Pitfalls.** Mistakes we hit in `cldk-mcp.ipynb`. For example, a short name like `init` is ambiguous, so always use the full signature.
- **`help()`.** CLDK's docstrings are detailed. The model can call `help(an.some_method)` to learn anything the card leaves out. MCP tools cannot offer this.

In [19]:
# The system prompt. Its wording is what the model learns CLDK from, so edit it with care.
SYSTEM = r"""You triage static-analysis alerts on a Python web application by writing Python.

## How to act
Each reply contains exactly one fenced block that starts with ```python. The harness runs it in a
persistent Python kernel and sends back what it printed, or the traceback if it failed. Variables
and imports persist between your scripts. Print everything you want to see; nothing else comes back.
When you have the answer, reply with no code block. That reply ends the run.

## What is already in the kernel
`an` is a CLDK 2.0 analysis of the application (3,710 callables), built with symbol table, call
graph and per-callable data dependence. `nx` is networkx. File paths are relative to the
application root, e.g. "testcode/BenchmarkTest00099.py". Line numbers are 1-based file lines.

## The CLDK calls you need
hit = an.locate(path, line)
    The callable that holds a line. hit.callable is None at module scope; otherwise
    hit.callable.signature  e.g. "testcode.BenchmarkTest00099.init.BenchmarkTest00099_post"
    hit.callable.name       e.g. "BenchmarkTest00099_post"
    hit.module.module_name, hit.diagnostics (why a line resolved to no callable)

an.get_method_bodies([signature, ...]) -> dict signature -> source text
    The source of each callable. A missing key means no such callable.

an.callers_of(signature) -> list of nodes ; an.callees_of(signature) -> list of nodes
    Who calls a callable, and what it calls (including library calls such as
    builtins.Cursor.execute). Each node has .callable (signature), .file, .line, .name.

page = an.get_ddg(signature)
    The data-dependence graph inside one callable. page.edges is a list of edges with
    .src and .dst (node ids) and .var (the variable or expression that carries the value).
    A node id ends in "@<line>:<col>", e.g. "can://owasp/python/.../BenchmarkTest00099_post()@31:2".
    Take the line with int(node_id.split("@")[-1].split(":")[0]).
    An edge src -> dst means the value defined on line src is used on line dst.

help(an.<method>)
    CLDK's own documentation of any method. Use it before calling a method not listed here.

## Pitfalls
- Pass full signatures. A bare name such as "init" matches 1,230 callables and raises AmbiguousName.
- For a local variable, get_ddg is the dependable call. resolve_value and flows_to_argument
  resolve parameters, globals and captures only.
- An alert's line is where the scanner fired. Read the source to find where the request value
  enters (request.form.get, request.args.get, request.cookies, request.headers) and where it is used.

## The answer
End with a reply that has no code and states, in this order:
VERDICT: TRUE POSITIVE or FALSE POSITIVE
EVIDENCE: the chain of lines (and the variable on each hop) from the request value to the
flagged line, or the reason no chain exists.
"""

print(len(SYSTEM.split()), "words")

407 words


### Check the card against CLDK

The card makes claims about CLDK. This cell tests each claim on the real object. If a future CLDK release changes a name or a field, this cell fails before the model is given wrong instructions.

In [20]:
# Every method the card names must exist.
for name in ["locate", "get_method_bodies", "callers_of", "callees_of", "get_ddg"]:
    assert callable(getattr(an, name)), name

# locate(): the fields the card promises.
hit = an.locate("testcode/BenchmarkTest00099.py", 46)
sig = hit.callable.signature
assert hit.callable.name and hit.module.module_name and hit.diagnostics == []

# locate() at module scope gives no callable.
assert an.locate("testcode/BenchmarkTest00099.py", 3).callable is None

# get_method_bodies(): returns a dict keyed by signature.
assert sig in an.get_method_bodies([sig])

# callers_of(): each node has these four fields.
node = an.callers_of(sig)[0]
assert node.callable and node.file and node.line and node.name

# get_ddg(): each edge has src, dst and var, and the line sits after the "@".
edge = an.get_ddg(sig).edges[0]
assert edge.src and edge.dst and edge.var
assert int(edge.src.split("@")[-1].split(":")[0]) > 0

print("the API card matches CLDK", importlib.metadata.version("cldk"))

the API card matches CLDK 2.0.0


## 4. Connect the model

We use **`qwen3.8-27b`** on the local server at `localhost:8080`.

- The server speaks the OpenAI chat API, so a plain HTTP request is enough.
- We send no `tools` list. A CodeAct model writes code instead of calling tools.
- Settings match HW1: temperature 0, seed 0, thinking off.
- To use another OpenAI-compatible server, set `LOCAL_MODEL_URL`.

In [21]:
import json
import urllib.request

# Where the model server lives, and how we call it.
SERVER = os.environ.get("LOCAL_MODEL_URL", "http://localhost:8080").rstrip("/")
MODEL = {"name": "qwen3.8-27b", "temperature": 0.0, "seed": 0, "max_tokens": 1500}


def chat(messages: list[dict]) -> tuple[str, dict]:
    """Send the conversation to the model. Return its reply and the token counts."""
    body = {
        "model": MODEL["name"],
        "messages": messages,
        "stream": False,
        "temperature": MODEL["temperature"],
        "seed": MODEL["seed"],
        "max_tokens": MODEL["max_tokens"],
        # Turn off Qwen's hidden "thinking" so we only get the visible reply.
        "chat_template_kwargs": {"enable_thinking": False},
    }
    req = urllib.request.Request(
        f"{SERVER}/v1/chat/completions",
        data=json.dumps(body).encode(),
        headers={"Content-Type": "application/json"},
    )
    with urllib.request.urlopen(req, timeout=600) as resp:
        data = json.load(resp)

    reply = data["choices"][0]["message"]["content"]
    usage = {"input": data["usage"]["prompt_tokens"], "output": data["usage"]["completion_tokens"]}
    return reply, usage


# Smoke test: the server is up and answers.
print(chat([{"role": "user", "content": "Reply with the single word ready."}])[0])

ready


## 5. The agent loop

The loop follows lecture 4's six steps (notes, section 4.3):

1. Send the conversation to the model.
2. The model replies with a script.
3. Run the script in the kernel.
4. Collect what it printed, or its traceback.
5. Add that output to the conversation and go back to step 1.
6. If a reply has no code, it is the final answer. Stop.

`MAX_TURNS` limits how many times we call the model. It does **not** limit how long one script runs. That is one of the costs of code actions from lecture 4.

In [22]:
MAX_TURNS = 8


def codeact(task: str, show: bool = True) -> dict:
    """Run one CodeAct episode. Return the answer, the full conversation and the token usage."""
    namespace = new_namespace()  # a fresh kernel for each episode
    messages = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": task},
    ]
    usage = {"input": 0, "output": 0}

    for turn in range(1, MAX_TURNS + 1):
        # Steps 1 and 2: ask the model, get its reply.
        reply, u = chat(messages)
        usage = {k: usage[k] + u[k] for k in usage}
        messages.append({"role": "assistant", "content": reply})

        # Step 6: no code in the reply means it is the final answer.
        script = extract(reply)
        if script is None:
            if show:
                print(f"--- turn {turn}: final answer\n{reply}")
            return {"answer": reply, "turns": turn, "usage": usage, "messages": messages}

        # Steps 3 and 4: run the script and collect its output.
        observation = run(script, namespace)
        if show:
            print(f"--- turn {turn}: script\n{script}--- observation\n{observation}")

        # Step 5: send the output back as the next message.
        messages.append({"role": "user", "content": "Observation:\n" + observation})

    # Out of turns without a final answer.
    if show:
        print(f"--- stopped after {MAX_TURNS} turns without a final answer")
    return {"answer": None, "turns": MAX_TURNS, "usage": usage, "messages": messages}

### A true positive

Bandit flags rule **B608** (SQL built from a string) at line 46 of `BenchmarkTest00099.py`.

In `cldk-mcp.ipynb` we found the answer by hand: the request value on line 31 reaches the SQL string on line 46, through `param`, `conf52528.get` and `bar`.

The agent gets only what Bandit gives: the file, the line and the rule.

In [23]:
# The same task text is used for both alerts.
TASK = (
    "Bandit reports B608 (possible SQL injection through string-based query construction) "
    "at line {line} of {path}. Is it a true positive?"
)

tp = codeact(TASK.format(path="testcode/BenchmarkTest00099.py", line=46))
print(f"\n{tp['turns']} turns, {tp['usage']['input']} input and {tp['usage']['output']} output tokens")

--- turn 1: script
hit = an.locate("testcode/BenchmarkTest00099.py", 46)
print(hit)
print(hit.callable)
if hit.callable:
    print(hit.callable.signature)
--- observation
body=BodyRef(id='can://owasp/python/testcode/BenchmarkTest00099.py/init(app)/BenchmarkTest00099_post()@46:2', kind='statement', span=Span(start=(46, 2), end=(46, 64), bytes=(1456, 1518)), callee=None) node_id='can://owasp/python/testcode/BenchmarkTest00099.py/init(app)/BenchmarkTest00099_post()@46:2' callable=CallableRef(signature='testcode.BenchmarkTest00099.init.BenchmarkTest00099_post', name='BenchmarkTest00099_post', class_signature=None) type=None module=ModuleRef(path='testcode/BenchmarkTest00099.py', module_name='BenchmarkTest00099') source='def BenchmarkTest00099_post():\n\t\tRESPONSE = ""\n\n\t\tparam = request.form.get("BenchmarkTest00099")\n\t\tif not param:\n\t\t\tparam = ""\n\n\t\timport configparser\n\t\t\n\t\tbar = \'safe!\'\n\t\tconf52528 = configparser.ConfigParser()\n\t\tconf52528.add_section(\'secti

### A false positive

`BenchmarkTest00011.py`, B608 at line 47.

Here the SQL string is a constant with a `?` placeholder. The request value is passed separately, as a parameter to `execute` on line 50. So it never reaches the string on line 47.

In [24]:
fp = codeact(TASK.format(path="testcode/BenchmarkTest00011.py", line=47))
print(f"\n{fp['turns']} turns, {fp['usage']['input']} input and {fp['usage']['output']} output tokens")

--- turn 1: script
hit = an.locate("testcode/BenchmarkTest00011.py", 47)
print(hit)
print(hit.callable)
if hit.callable:
    print(hit.callable.signature)
--- observation
body=BodyRef(id='can://owasp/python/testcode/BenchmarkTest00011.py/init(app)/BenchmarkTest00011_post()@47:2', kind='statement', span=Span(start=(47, 2), end=(47, 56), bytes=(1520, 1574)), callee=None) node_id='can://owasp/python/testcode/BenchmarkTest00011.py/init(app)/BenchmarkTest00011_post()@47:2' callable=CallableRef(signature='testcode.BenchmarkTest00011.init.BenchmarkTest00011_post', name='BenchmarkTest00011_post', class_signature=None) type=None module=ModuleRef(path='testcode/BenchmarkTest00011.py', module_name='BenchmarkTest00011') source='def BenchmarkTest00011_post():\n\t\tRESPONSE = ""\n\n\t\timport urllib.parse\n\t\tparam = urllib.parse.unquote_plus(request.cookies.get("BenchmarkTest00011", "noCookieValueSupplied"))\n\n\t\tbar = "This should never happen"\n\t\tif \'should\' in bar:\n\t\t\tbar = param\n\n\

### What the agent did

Both answers are correct. They match the answer key.

**True positive (00099), 4 turns**

1. Located line 46.
2. Printed every data-dependence edge of the function.
3. Printed the source.
4. Answered with the chain 31 → 41 → 42 → 46, through `param`, `conf52528.get` and `bar`. This is the same chain we found by hand.

One slip: in step 3 it numbered the source from 1, but the function starts at line 28. Its answer still used the right line numbers, taken from the graph.

**False positive (00011), 3 turns**

1. Located line 47.
2. Printed every data-dependence edge.
3. Answered: `bar` (line 43) only reaches `execute` (line 50), and the SQL string on line 47 is a constant.

**What to notice**

- **The agent printed raw node ids.** The card shows how to turn them into line numbers, but the agent never did. So both edge listings hit the 6,000-character cap and were cut. The kept part was enough here. On a bigger function it might not be.
- **The prompt teaches the calls, but the model decides how to use them.** If you need a certain kind of evidence, say so in the prompt, or check the transcript before trusting the answer.

## 6. What the agent's code can reach

The MCP server only offered four tools, and it refused paths outside the checkout. None of the tools could read an arbitrary file.

The kernel has no such limit. The model's code runs with all the rights of this notebook. It can call `open()` on any file we can read.

The OWASP checkout even ships its **answer key**, `expectedresults-0.1.csv`. We open it by hand below, but any script from the model could do the same.

In [11]:
# A hand-written script, run exactly as a model's script would be.
print(
    run(
        """
import csv
for row in csv.reader(open("cldk-demo/owasp/expectedresults-0.1.csv")):
    if row[0] in ("BenchmarkTest00011", "BenchmarkTest00099"):
        print(row)
""",
        new_namespace(),
    )
)

['BenchmarkTest00011', 'sqli', 'false', '89']
['BenchmarkTest00099', 'sqli', 'true', '89']



Our two runs did not open the key. But nothing in the harness would have stopped them.

Two fixes, neither of which is a prompt change:

- **Move the key out of the checkout** before the agent runs.
- **Run the kernel in a sandbox** that can only see the checkout and the model server. That is the OpenShell part of this lecture.

We can also check afterwards. The next cell searches every script the model sent for the key's file name.

In [12]:
for name, result in [("00099", tp), ("00011", fp)]:
    # Every script the model sent during this run.
    scripts = [extract(m["content"]) or "" for m in result["messages"] if m["role"] == "assistant"]
    named = any("expectedresults" in s for s in scripts)
    print(name, "named the answer key" if named else "never named the answer key")

00099 never named the answer key
00011 never named the answer key


## Takeaway

- **MCP:** four fixed questions, and every argument is checked.
- **CodeAct:** the agent can ask CLDK anything, combine calls in one script, and read CLDK's docs with `help()`.
- **The cost:** the harness no longer sees each operation. A sandbox has to limit what the code can do.